# RAG Helpers Demonstration

This notebook demonstrates the main functions from `rag_helpers.py` that allow you to:
1. Connect to RAG/ChromaDB stored in Google Cloud Storage
2. Query document collections using natural language
3. Retrieve relevant documents with metadata and similarity scores

## Functions Demonstrated:
- `get_rag_connection()` - Sets up RAG connectivity in GCP Storage
- `get_chroma_db()` - Returns chroma_db_context for querying
- `chroma_db_context.query()` - Performs semantic search queries
- `query_rag_texts()` - Convenience function for one-step text retrieval
- `store_query_in_chromadb()` - Store queries in ChromaDB with embeddings


In [1]:
# Setup: Configure credentials and environment
import os
from pathlib import Path
from dotenv import load_dotenv

# Method 1: Load .env file from the same directory as this notebook
env_file = Path.cwd() / '.env'
if env_file.exists():
    load_dotenv(env_file, override=True)
    print(f"✓ Loaded .env from: {env_file}")
else:
    print(f"⚠ .env file not found at: {env_file}")

# Method 2: If GOOGLE_APPLICATION_CREDENTIALS not set or points to Docker path, try to find credentials
current_creds = os.getenv('GOOGLE_APPLICATION_CREDENTIALS', '')
if not current_creds or current_creds.startswith('/workspace'):
    # Try relative path: go up from src/rag to project root, then to secrets
    project_root = Path.cwd().parent.parent if Path.cwd().name == 'rag' else Path.cwd()
    
    # Try gcs-key.json first (standard name)
    creds_path = project_root / 'secrets' / 'gcs-key.json'
    
    # If not found, try the new credentials file name
    if not creds_path.exists():
        creds_path = project_root / 'secrets' / 'stock-busters-cs115-122896c71e74.json'
    
    if creds_path.exists():
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(creds_path.resolve())
        print(f"✓ Auto-set GOOGLE_APPLICATION_CREDENTIALS to: {creds_path.resolve()}")
    else:
        # Try absolute path (Windows) - check both filenames
        abs_creds = Path(r"C:\Users\eilke\CSCI115-AI-Agent\secrets\gcs-key.json")
        if not abs_creds.exists():
            abs_creds = Path(r"C:\Users\eilke\CSCI115-AI-Agent\secrets\stock-busters-cs115-122896c71e74.json")
        
        if abs_creds.exists():
            os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(abs_creds)
            print(f"✓ Auto-set GOOGLE_APPLICATION_CREDENTIALS to: {abs_creds}")
        elif current_creds.startswith('/workspace'):
            print(f"⚠ Docker path detected ({current_creds}), but local credentials not found")
            print("   Please set GOOGLE_APPLICATION_CREDENTIALS manually or ensure credentials exist")

# Verify credentials are configured
if os.getenv('GOOGLE_APPLICATION_CREDENTIALS'):
    creds_file = Path(os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
    if creds_file.exists():
        print(f"✓ Credentials file verified: {creds_file}")
    else:
        print(f"⚠ Credentials file not found at: {creds_file}")
else:
    print("⚠ GOOGLE_APPLICATION_CREDENTIALS not set. Please set it manually or in .env file")

# Check GCS_BUCKET_NAME
if os.getenv('GCS_BUCKET_NAME'):
    print(f"✓ GCS_BUCKET_NAME: {os.getenv('GCS_BUCKET_NAME')}")
else:
    print("⚠ GCS_BUCKET_NAME not set. Please set it in .env file or as environment variable")


✓ Loaded .env from: C:\Users\eilke\CSCI115-AI-Agent\src\rag\.env
✓ Auto-set GOOGLE_APPLICATION_CREDENTIALS to: C:\Users\eilke\CSCI115-AI-Agent\secrets\gcs-key.json
✓ Credentials file verified: C:\Users\eilke\CSCI115-AI-Agent\secrets\gcs-key.json
✓ GCS_BUCKET_NAME: stock-busters-chroma-bucket


In [2]:
# Import the main functions
# Note: rag_helpers.py will automatically load .env file via load_dotenv()
# This may set GOOGLE_APPLICATION_CREDENTIALS to Docker path, so we override if needed
from rag_helpers import (
    get_rag_connection,
    get_chroma_db,
    query_rag_texts,
    store_query_in_chromadb,
)

# Override credentials path if it's a Docker path and we have local credentials
current_creds = os.getenv('GOOGLE_APPLICATION_CREDENTIALS', '')
if current_creds.startswith('/workspace'):
    # Try to find local credentials
    project_root = Path.cwd().parent.parent if Path.cwd().name == 'rag' else Path.cwd()
    creds_path = project_root / 'secrets' / 'gcs-key.json'
    if not creds_path.exists():
        creds_path = project_root / 'secrets' / 'stock-busters-cs115-122896c71e74.json'
    if creds_path.exists():
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(creds_path.resolve())
        print(f"✓ Overrode credentials path to: {creds_path.resolve()}")
    else:
        # Try absolute path
        abs_creds = Path(r"C:\Users\eilke\CSCI115-AI-Agent\secrets\gcs-key.json")
        if abs_creds.exists():
            os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(abs_creds)
            print(f"✓ Overrode credentials path to: {abs_creds}")

print("✓ Imports successful!")


✓ Overrode credentials path to: C:\Users\eilke\CSCI115-AI-Agent\secrets\gcs-key.json
✓ Imports successful!


## Function 1: `get_rag_connection()`

This function establishes RAG connectivity by downloading ChromaDB files from Google Cloud Storage and initializing a local ChromaDB client. It handles all the setup automatically.

**What it does:**
- Downloads all ChromaDB files from GCS bucket (prefix: "chromadb")
- Stores them in a local temporary directory
- Creates a ChromaDB PersistentClient connection
- Caches the connection for reuse

**Note:** This function is automatically called by `get_chroma_db()` if not already set up, so you can skip calling it explicitly if you prefer.


In [3]:
# Set up connection explicitly (optional - get_chroma_db() will call this automatically)
# You can specify a collection name, or use the default from environment
get_rag_connection()

# Or with a specific collection:
# get_rag_connection(collection_name="my_collection")


Downloaded 6 files from GCS
Connected to RAG/ChromaDB (collection: stocks_rag_v1)


## Function 2: `get_chroma_db()`

This function returns a `chroma_db_context` object that can be used to query ChromaDB collections. It automatically ensures the connection is set up by calling `get_rag_connection()` if needed.

**Returns:** A `chroma_db_context` object with a `.query()` method


In [4]:
# Get chroma_db_context
chroma_db = get_chroma_db()

# Or with a specific collection:
# chroma_db = get_chroma_db(collection_name="my_collection")

print(f"✓ Got chroma_db_context: {type(chroma_db)}")
print(f"✓ Has query method: {hasattr(chroma_db, 'query')}")


✓ Got chroma_db_context: <class 'types.SimpleNamespace'>
✓ Has query method: True


## Function 3: `chroma_db_context.query()`

This is the main query method that performs semantic search on ChromaDB collections. It converts your natural language query to an embedding and finds the most similar documents.

**Parameters:**
- `query_string`: The natural language query/question (e.g., "What is ROE?")
- `collection`: Optional collection name (uses default if not specified)
- `k`: Number of results to return (default: 4, max: 50)

**Returns:** List of dictionaries with:
- `id`: Document ID in ChromaDB
- `document`: The document text content
- `metadata`: Dictionary of document metadata (source, page, etc.)
- `distance`: Similarity distance (lower = more similar)


In [5]:
# Query the default collection
query = "What is ROE?"
results = chroma_db.query(query, k=3)

print(f"Query: '{query}'")
print(f"Found {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Metadata: {result['metadata']}")
    print(f"  Document (first 200 chars): {result['document'][:200]}...")
    print()


Query: 'What is ROE?'
Found 3 results:

Result 1:
  Distance: 0.6046
  Metadata: {'source': '/workspace/data/Output_explanation.csv#feature=roe', 'chunk_index': 0, 'total_chunks': 1, 'content_hash': 'd434b6d149cb7c3f6319794717994c9c'}
  Document (first 200 chars): Feature: roe Full Name or Formula: Return on Equity = Net Income / Shareholders' Equity Meaning: Measures how efficiently a company generates profits from shareholder equity. Interpretation or Signal:...

Result 2:
  Distance: 0.7651
  Metadata: {'content_hash': '1254c8e876bf216a0effed7fedd64a72', 'source': '/workspace/data/PrinciplesofFinance.pdf#chapter=6:Measures_of_Financial_Health', 'total_chunks': 51, 'chunk_index': 34}
  Document (first 200 chars): The company needs to use its assets and operations efficiently to increase profit. To assist with profit goal attainment, company revenues need to outweigh expenses. Industry standards can dictate wha...

Result 3:
  Distance: 0.7840
  Metadata: {'source': '/workspace/data/F

In [6]:
# Query a specific collection (if different from default)
# results = chroma_db.query("What is P/E ratio?", collection="financial_terms", k=5)

# Get more results
results = chroma_db.query("What is momentum?", k=5)
print(f"Query: 'What is momentum?'")
print(f"Found {len(results)} results\n")

# Access individual results
for i, r in enumerate(results, 1):
    print(f"{i}. Distance: {r['distance']:.4f}")
    print(f"   {r['document'][:150]}...")
    print()


Query: 'What is momentum?'
Found 5 results

1. Distance: 0.5851
   But we want to be clear: this book is about stock-selection momentum. But in order to really understand how to build any active investing strategy, we...

2. Distance: 0.6291
   Why should value and momentum approaches be mutually exclusive? Indeed, a key aspect of the scientific method is to preserve the freedom to doubt, for...

3. Distance: 0.6399
   64 ] = -0 . 29. The more negative the FIP, the better. The FIP algorithm separates high momentum stocks into those that have more continuous price pat...

4. Distance: 0.6462
   Enter the symbol of the stock you're interested in. After you enter the stock symbol into the blank, click on the Company Charts button. View the data...

5. Distance: 0.6470
   From an equilibrium perspective, not everyone can follow our strategy because for every stock we buy, there is a seller on the other side of the trade...



## Function 4: `query_rag_texts()` - Convenience Function

This is a one-step convenience function that connects to RAG, runs a query, and returns only the document texts (no metadata or distances). Perfect for quick retrieval when you just need the text content.

**Parameters:**
- `query_string`: Natural language query
- `collection_name`: Optional collection name
- `k`: Number of results (default: 4)

**Returns:** List of document text strings (most similar first)


In [7]:
# Simplest way to get texts directly (like get_gcs_csv pattern)
texts = query_rag_texts("What is ROE?", k=3)

print(f"Found {len(texts)} texts:\n")
for i, text in enumerate(texts, 1):
    print(f"Text {i} (first 200 chars):")
    print(f"{text[:200]}...")
    print()


Found 3 texts:

Text 1 (first 200 chars):
Feature: roe Full Name or Formula: Return on Equity = Net Income / Shareholders' Equity Meaning: Measures how efficiently a company generates profits from shareholder equity. Interpretation or Signal:...

Text 2 (first 200 chars):
The company needs to use its assets and operations efficiently to increase profit. To assist with profit goal attainment, company revenues need to outweigh expenses. Industry standards can dictate wha...

Text 3 (first 200 chars):
You could create an entire book just dedicated to financial ratios. Some fundamental analysts love some, like return on equity, while others find them to be misleading. At the end of a game, there's n...



In [8]:
# Query with specific collection
# texts = query_rag_texts("Explain P/E ratio", collection_name="financial_terms", k=5)

# Example: Get texts and process them
texts = query_rag_texts("What is earnings per share?", k=2)
for t in texts:
    print(t[:150])
    print("-" * 80)


When that happens, it's called a net loss . At its simplest level, net profit is operating profit minus everything else. Wonderful. So, what's in it f
--------------------------------------------------------------------------------
Wonderful. So, what's in it for me? Everything you really need to know, though, is right on the income statement, including: ✓ Basic earnings per shar
--------------------------------------------------------------------------------


## Function 5: `store_query_in_chromadb()` - Store Queries

This function allows you to store queries (like "P/E ratio") in ChromaDB collections. It automatically creates embeddings for the query and stores it with optional metadata.

**Parameters:**
- `query`: Query string to store (e.g., "P/E ratio", "What is ROE?")
- `collection_name`: Optional collection name (uses default if not specified)
- `query_id`: Optional query ID (auto-generated if not provided)
- `metadata`: Optional metadata dictionary (e.g., {"timestamp": "...", "user": "..."})
  - If not provided, a default metadata `{"stored_by": "rag_helpers"}` will be used
  - ChromaDB requires non-empty metadata dictionaries
- `upload_to_gcs`: If True, uploads updated ChromaDB files back to GCS (default: False)

**Returns:** Dictionary with stored count, query_id, collection name, and embedding model used


In [9]:
# Store a query (like your teammate wants)
result = store_query_in_chromadb(
    query="P/E ratio",
    collection_name="queries",
    metadata={"timestamp": "2024-01-15", "source": "user_input"}
)

print(f"✓ Stored query '{result['query_id']}' in collection '{result['collection']}'")
print(f"✓ Used embedding model: {result['embedding_model']}")


Downloaded 6 files from GCS
Connected to RAG/ChromaDB (collection: queries)


✓ Stored query 'query_p/e_ratio' in collection 'queries'
✓ Used embedding model: BAAI/bge-small-en-v1.5


In [10]:
# Store multiple queries with custom IDs
queries = ["momentum", "P/E ratio", "ROE"]
for q in queries:
    result = store_query_in_chromadb(
        query=q,
        query_id=f"query_{q.lower().replace('/', '_').replace(' ', '_')}",
        collection_name="queries",
        metadata={"type": "financial_ratio"}
    )
    print(f"✓ Stored query: {q} (ID: {result['query_id']})")

# Now query to verify the queries were stored
chroma_db = get_chroma_db(collection_name="queries")
results = chroma_db.query("P/E ratio", collection="queries", k=3)
if results:
    print(f"\n✓ Query test successful! Found {len(results)} stored query(ies):")
    for r in results:
        print(f"  - {r['document']} (ID: {r['id']})")


✓ Stored query: momentum (ID: query_momentum)
✓ Stored query: P/E ratio (ID: query_p_e_ratio)
✓ Stored query: ROE (ID: query_roe)

✓ Query test successful! Found 3 stored query(ies):
  - P/E ratio (ID: query_p/e_ratio)
  - P/E ratio (ID: query_p_e_ratio)
  - ROE (ID: query_roe)


In [11]:
# Store query and upload to GCS (persists changes)
# Note: This will upload the updated ChromaDB files back to GCS
# result = store_query_in_chromadb(
#     query="P/E ratio",
#     collection_name="queries",
#     upload_to_gcs=True  # Set to True to persist changes to GCS
# )


## Complete Usage Pattern 

1. `get_rag_connection()` - Sets up connection
2. `get_chroma_db()` - Returns chroma_db_context
3. `chroma_db_context.query(string, collection)` - Query and get results


In [12]:
# Step 1: Set up connection (optional - get_chroma_db() does this automatically)
get_rag_connection()

# Step 2: Get chroma_db_context
chroma_db = get_chroma_db()

# Step 3: Query using chroma_db_context.query(string, collection)
results = chroma_db.query("What is ROE?", collection=None, k=4)

# Process results
print(f"Query returned {len(results)} results:\n")
for r in results:
    print(f"Document: {r['document'][:100]}...")
    print(f"Distance: {r['distance']:.4f}")
    print(f"Metadata: {r['metadata']}")
    print()


Query returned 4 results:

Document: Feature: roe Full Name or Formula: Return on Equity = Net Income / Shareholders' Equity Meaning: Mea...
Distance: 0.6046
Metadata: {'total_chunks': 1, 'content_hash': 'd434b6d149cb7c3f6319794717994c9c', 'chunk_index': 0, 'source': '/workspace/data/Output_explanation.csv#feature=roe'}

Document: The company needs to use its assets and operations efficiently to increase profit. To assist with pr...
Distance: 0.7651
Metadata: {'content_hash': '1254c8e876bf216a0effed7fedd64a72', 'source': '/workspace/data/PrinciplesofFinance.pdf#chapter=6:Measures_of_Financial_Health', 'chunk_index': 34, 'total_chunks': 51}

Document: You could create an entire book just dedicated to financial ratios. Some fundamental analysts love s...
Distance: 0.7840
Metadata: {'source': '/workspace/data/Fundamental Analysis for Dummies.pdf#chapter=8:Using_Financial_Ratios_to_Pinpoint_Investments', 'total_chunks': 26, 'content_hash': '75c54bd3e8e2918c9e49cf84b50c7b82', 'chunk_index':

## Summary

**Main Functions:**
- `get_rag_connection(collection_name=None)` - Sets up RAG connectivity in GCP Storage
- `get_chroma_db(collection_name=None)` - Returns chroma_db_context
- `chroma_db_context.query(query_string, collection=None, k=4)` - Query and get results
- `query_rag_texts(query_string, collection_name=None, k=4)` - Convenience function for text-only results
- `store_query_in_chromadb(query, collection_name=None, query_id=None, metadata=None, upload_to_gcs=False)` - Store queries in ChromaDB

**Key Features:**
- Works independently (no need to run `--serve`)
- Automatically downloads ChromaDB files from GCS
- Caches connections for efficiency
- Returns rich results with documents, metadata, and similarity scores
- Can store queries (like "P/E ratio") with automatic embedding generation
